# Oversampling

### <i> Using SDV Framework

<br>
<br>

`SDV` learns the overall data distribution and generates synthetic records from that learned distribution. You have a wide variety of options (Synthesizer algorithms) you can work with when deciding how you want to create more data:

</i>
<br>

|Synthesizer|Approach|
|---|---|
|GaussianCopulaSynthesizer| statistical/correlation-based approach (REALLY robust)|
|CTGAN| neural network-based|
|TVAE| variational autoencoder approach (good for noise)|
|CopulaGAN| GAN + copula approach (handles huge ranges in values decently well)|


<i>



---

### TVAE

In [ ]:
from sdv.metadata import Metadata
from sdv.single_table import TVAESynthesizer

import pandas as pd

# Load a new dataframe.
df = pd.read_csv("../../houses.csv")

df.drop(["id", "date", "lat", "long", 'zipcode'], axis=1, inplace=True)


# Create the metadata again
metadata = Metadata.detect_from_dataframe(
    data=df
)

# Place the metadata into the TVAE model approach
synthesizer = TVAESynthesizer(
    metadata
)

# Set and fit the data into our dataframe.
# TVAE takes more time than Copula,
# but considerably less than CTGAN (4-5 minutes)
synthesizer.fit(df)

# Make 5k rows using Gaussian Copula
synthetic_df = synthesizer.sample(
    num_rows=5000
)

c:\Users\tuomas.valtanen\dataeng_advda2026_lectures\DataEngineering_AdvDA_2026\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [3]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Let's make two histplots for each column. One for the Original dataset, the other for CTGAN.
# for column in df.columns:

#     # Create the canvas and create
#     fig, axes = plt.subplots(1, 2, figsize=(12, 5))
#     sns.histplot(df[column], ax=axes[0], color="red")
#     sns.histplot(synthetic_data[column], ax=axes[1], color="gray")

#     axes[0].set_title("Original")
#     axes[1].set_title("TVAE Synthetic")


<i>
Visually, this looks like the best fit for the dataset. The shapes of each distribution seems about right!

<br>

We visually checked and inspected each column's shape individually, but that's not enough. We need to check the fit for the pairs as well. This neat tool does both:

###  Using `SDV` evaluate Quality

In [4]:
# Let's use the evaluate_quality function in SDV's library
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    real_data=df,
    synthetic_data=synthetic_df,
    metadata=metadata
)

# The higher the percentile, the better the fit. 
quality_report

# Column shape: Compares real data vs synthetic data 

C:\Users\tuomas.valtanen\AppData\Local\Temp\ipykernel_26808\2975143818.py:2: FutureWarning: The evaluation functions are now accessible via the 'sdv.evaluation' module.
  from sdv.evaluation.single_table import evaluate_quality


Generating report ...

(1/4) Evaluating Column Shapes: |██████████| 16/16 [00:00<00:00, 158.86it/s]|
Column Shapes Score: 92.86%

(2/4) Evaluating Column Pair Trends: |██████████| 120/120 [00:00<00:00, 199.71it/s]|
Column Pair Trends Score: 95.21%

(3/4) Evaluating Cardinality: N/A
This property does not apply to single-table data.

(4/4) Evaluating Intertable Trends: N/A
This property does not apply to single-table data.

Overall Score (Average): 94.04%



---

### Studying and Comparing TVAE with the Original

<i> Never take synthetic data at face value, always inspect if it reflects reality first.

In [7]:
# switching to decimal notation for the variance checks
pd.set_option('display.float_format', lambda x: "%.2f" % x)

In [8]:
# We can compare the statistics between the two dataframes by using `.describe()`
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
count,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00
mean,540088.14,3.37,2.11,2079.90,15106.97,1.49,0.01,0.23,3.41,7.66,1788.39,291.51,1971.01,84.40,1986.55,12768.46
std,367127.20,0.93,0.77,918.44,41420.51,0.54,0.09,0.77,0.65,1.18,828.09,442.58,29.37,401.68,685.39,27304.18
min,75000.00,0.00,0.00,290.00,520.00,1.00,0.00,0.00,1.00,1.00,290.00,0.00,1900.00,0.00,399.00,651.00
25%,321950.00,3.00,1.75,1427.00,5040.00,1.00,0.00,0.00,3.00,7.00,1190.00,0.00,1951.00,0.00,1490.00,5100.00
50%,450000.00,3.00,2.25,1910.00,7618.00,1.50,0.00,0.00,3.00,7.00,1560.00,0.00,1975.00,0.00,1840.00,7620.00
75%,645000.00,4.00,2.50,2550.00,10688.00,2.00,0.00,0.00,4.00,8.00,2210.00,560.00,1997.00,0.00,2360.00,10083.00
max,7700000.00,33.00,8.00,13540.00,1651359.00,3.50,1.00,4.00,5.00,13.00,9410.00,4820.00,2015.00,2015.00,6210.00,871200.00


In [9]:
synthetic_df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,553858.82,3.29,2.12,2125.99,10693.48,1.50,0.00,0.05,3.23,7.74,1852.04,296.81,1969.47,69.87,2041.06,9522.05
std,380739.49,0.65,0.74,791.78,15130.74,0.53,0.03,0.46,0.43,1.07,752.37,410.12,28.82,365.82,663.72,11841.63
min,75000.00,1.00,0.16,699.00,520.00,1.00,0.00,0.00,3.00,5.00,680.00,0.00,1900.00,0.00,583.00,651.00
25%,329881.75,3.00,1.75,1517.00,5053.75,1.00,0.00,0.00,3.00,7.00,1310.00,0.00,1951.00,0.00,1540.75,5301.50
50%,476556.50,3.00,2.25,1935.00,7616.00,1.50,0.00,0.00,3.00,8.00,1623.00,1.00,1971.00,0.00,1911.50,7404.50
75%,635761.50,4.00,2.50,2611.25,10942.25,2.00,0.00,0.00,3.00,8.00,2225.25,548.00,1993.00,0.00,2419.50,9715.25
max,3992455.00,6.00,4.52,5084.00,305055.00,3.00,1.00,4.00,5.00,12.00,6222.00,2692.00,2015.00,2003.00,5287.00,253827.00


In [10]:
synthetic_df

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
0,747221.00,4,2.50,3362,10912,2.00,0,0,4,8,2786,605,1975,0,3092,7913
1,606378.00,3,1.75,1501,3898,1.00,0,0,3,7,1277,410,1924,0,1379,4694
2,201005.00,3,2.00,1396,16122,1.00,0,0,3,7,1499,1,2001,0,1580,51649
3,377215.00,3,2.81,1830,12428,2.00,0,0,3,8,1633,2,1987,0,2266,11032
4,313070.00,3,2.50,1604,10975,1.00,0,0,3,7,1865,0,2001,0,1880,12710
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,738927.00,4,3.55,3530,12273,2.00,0,0,3,9,1090,0,1992,0,3470,11031
4996,263196.00,3,2.25,1803,13367,1.00,0,0,4,8,1676,1,1977,0,2270,18308
4997,351248.00,2,1.00,1312,9174,1.00,0,0,3,7,1257,0,1947,0,1448,7449
4998,764072.00,4,2.50,3209,19502,2.00,0,0,3,9,3759,0,1999,0,3180,8131


In [ ]:
synthetic_df.query("yr_renovated < yr_built")

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
35,607324.00,2,1.75,1940,4813,1.00,0,0,3,7,1209,1060,1923,1994,2148,5707
41,649022.00,4,2.50,3025,8837,1.00,0,0,4,8,1754,1324,1946,1994,2648,9042
62,566056.00,2,1.00,1517,6356,1.50,0,0,3,7,1445,1,1907,1978,1526,3069
68,576190.00,4,1.75,2349,9683,1.00,0,0,3,7,1209,1143,1952,1983,1158,7994
110,772172.00,3,1.75,1544,3367,1.50,0,0,3,8,1298,195,1927,1991,1800,4418
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4842,2097002.00,4,3.59,4727,14117,2.00,0,4,3,11,3928,0,1908,1984,3989,7921
4846,3037437.00,4,3.87,3814,7257,1.50,0,4,5,9,2335,1383,1921,1993,3427,4110
4870,835345.00,4,1.75,2392,10900,1.00,0,4,4,8,1486,1140,1949,1989,2892,8931
4873,563122.00,3,1.75,2575,8746,1.00,0,0,3,7,1339,784,1945,1986,1755,7786
